# Exercise 2 — Cleaning Café Cozy's Sales Data with pandas

**Time:** ~15 minutes

**Goal:** Use pandas to fix a messy sales export — the same kinds of problems you saw in Macarena's *Data Cleaning Cycle* slide.

**The dataset:** `dirty_sales.csv` — a real-looking export with all the common pain points:
- Missing values (blanks, `N/A`)
- Duplicate rows
- Inconsistent strings (`Latte`, `latte`, ` LATTE `, `Muffin` vs `Blueberry Muffin`)
- Wrong number format (`3,75` instead of `3.75` — German decimal)
- Mixed date formats (`2026-04-01` vs `02/04/2026`)
- Inconsistent country names (`Germany`, `DE`, `deutschland`)

## Step 1 — Load and inspect

`pandas` is the standard Python library for tabular data. A `DataFrame` is like a SQL table or an Excel sheet.

In [ ]:
import pandas as pd

df = pd.read_csv('dirty_sales.csv')
df.head(10)

In [ ]:
# How many rows / columns? What types?
df.info()

In [ ]:
# Missing values per column
df.isnull().sum()

## Step 2 — Remove duplicates

**TODO:** Drop exact duplicate rows. `df.drop_duplicates()` returns a new DataFrame.

In [ ]:
print(f"Before: {len(df)} rows")

# YOUR CODE HERE
df = df  # replace with drop_duplicates() call

print(f"After:  {len(df)} rows")

## Step 3 — Standardize product names

The `product` column has trailing spaces and mixed casing. We want one canonical form per product.

**TODO:** Strip whitespace, lowercase everything, then title-case it. Use `.str` accessor on the column.

Then map `'Muffin'` → `'Blueberry Muffin'` so they're not split into two products.

In [ ]:
print("Unique products before:", df['product'].unique())

# YOUR CODE HERE
df['product'] = df['product']  # apply .str.strip().str.lower().str.title()
df['product'] = df['product'].replace({'Muffin': 'Blueberry Muffin'})

print("Unique products after: ", df['product'].unique())

## Step 4 — Fix the price column

Some prices are written `3,75` (German format) instead of `3.75`. And `N/A` is there too. The column came in as text.

**TODO:** Replace `,` with `.`, then convert to a numeric type. Use `pd.to_numeric(..., errors='coerce')` to turn unparseable values into `NaN`.

In [ ]:
# YOUR CODE HERE
df['price_eur'] = df['price_eur'].astype(str).str.replace(',', '.', regex=False)
df['price_eur'] = pd.to_numeric(df['price_eur'], errors='coerce')

df['price_eur'].describe()

## Step 5 — Handle missing values

Some `quantity` rows are blank. Some `country` values are missing. Some `price_eur` rows became `NaN` in the last step.

**Decision time** — these are choices a data analyst makes every day:
- Quantity missing → assume 1 (single item)
- Price missing → drop the row (can't compute revenue without it)
- Country missing → fill with `'Unknown'`

**TODO:** apply each rule below.

In [ ]:
df['quantity'] = df['quantity'].fillna(1).astype(int)
df['country'] = df['country'].fillna('Unknown')
df = df.dropna(subset=['price_eur'])

df.isnull().sum()

## Step 6 — Standardize country names

**TODO:** Map every variant of Germany to `'Germany'`. Leave Austria as-is.

*Hint:* lowercase, then map.

In [ ]:
country_map = {
    'germany':     'Germany',
    'de':          'Germany',
    'deutschland': 'Germany',
    'austria':     'Austria',
    'unknown':     'Unknown',
}

# YOUR CODE HERE
df['country'] = df['country'].str.lower().map(country_map)

df['country'].value_counts()

## Step 7 — Parse dates

Two formats live in the `date` column. `pd.to_datetime` is forgiving and can usually figure it out.

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=False)
df.dtypes

## Step 8 — Final check

What does our cleaned data look like?

In [ ]:
df.head()

In [ ]:
# Quick KPI preview — revenue per product
df['revenue'] = df['price_eur'] * df['quantity']
df.groupby('product')['revenue'].sum().sort_values(ascending=False)

### What you just practiced

| pandas operation | SQL equivalent |
|---|---|
| `df.drop_duplicates()` | `SELECT DISTINCT` |
| `df['col'].fillna(x)` | `COALESCE(col, x)` |
| `df['col'].str.lower()` | `LOWER(col)` |
| `df.groupby('col').sum()` | `GROUP BY col` |
| `df.dropna(subset=['col'])` | `WHERE col IS NOT NULL` |

Next: we'll combine SQL **and** pandas in one pipeline.